# Colab bootstrap

Run the cell below once after connecting to a new Colab runtime. It mounts Google Drive, clones or updates the repository, installs the project, and configures the paths used by the detection YAML.

In [7]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/terriljoel/retrieval-grounded-remote-sensing.git"
REPO_REF = "feat/object-detection-YOLO"  # Change to main after merging.
REPO_DIR = Path("/content/retrieval-grounded-remote-sensing")
SHARED_ROOT = Path("/content/drive/Othercomputers/My laptop/shared_resources")

drive.mount("/content/drive")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_REF], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"Repository path exists but is not a Git clone: {REPO_DIR}")
else:
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[annotation]"],
    check=True,
)

if not SHARED_ROOT.is_dir():
    raise FileNotFoundError(
        f"Shared resources were not found at {SHARED_ROOT}. "
        "Check the Google Drive computer and folder names."
    )

os.environ.update({
    "SHARED_RESOURCES_ROOT": str(SHARED_ROOT),
    "RAW_DATASET_ROOT": "/content/datasets/raw",
    "PROCESSED_DATASET_ROOT": "/content/datasets/processed",
    "MANIFEST_ROOT": str(SHARED_ROOT / "datasets" / "manifests"),
    "EXPERIMENT_OUTPUT_ROOT": str(SHARED_ROOT / "experiment_outputs"),
    "JOB_LOG_ROOT": str(SHARED_ROOT / "experiment_outputs" / "job_logs"),
    "INFERENCE_EXPORT_ROOT": str(SHARED_ROOT / "inference" / "object_detection_inference_new_remote_sensing_dataset_external-4"),
})

Path(os.environ["MANIFEST_ROOT"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["EXPERIMENT_OUTPUT_ROOT"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["JOB_LOG_ROOT"]).mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

print(f"Ready. Working directory: {Path.cwd()}")
print("The detector commands and annotation app are ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready. Working directory: /content/retrieval-grounded-remote-sensing
The detector commands and annotation app are ready.


In [8]:
%cd /content/retrieval-grounded-remote-sensing

!git pull --ff-only
!pip install -e ".[annotation]"

/content
Already up to date.
Obtaining file:///content/retrieval-grounded-remote-sensing
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for retrieval-grounded-remote-sensing (pyproject.toml) ... done
  Created wheel for retrieval-grounded-remote-sensing: filename=retrieval_grounded_remote_sensing-0.1.0-0.editable-py3-none-any.whl size=7499 sha256=5e3279f4975ea60c01a10e197d98f140f5494c3d30688eee77dcf7711944c851
  Stored in directory: /tmp/pip-ephem-wheel-cache-qktjlwhw/wheels/52/96/76/9ecc1b2d74cd8f5e164151eb125c540390d011574b114b11c1
Successfully built retrieval-grounded-remote-sensing
  Attempting uninstall: retrieval-grounded-remote-sensing
    Found existing installation: retrieval-grounded-remote-sensing 0.1.0
    Uninstalling retrieval-grounded-remote-sensing-0.1.0:
      Successfully uninstalled retrie

## below code to enter NVIDIA api key

In [9]:
from getpass import getpass
import os

nim_api_key = os.environ.get("NIM_API_KEY", "").strip()

if not nim_api_key:
    nim_api_key = getpass("NVIDIA NIM API key: ").strip()

if not nim_api_key.startswith("nvapi-"):
    raise ValueError("The supplied key is not a valid NVIDIA NIM API key")

os.environ["NIM_API_KEY"] = nim_api_key

print("NVIDIA NIM API key configured for this runtime.")

NVIDIA NIM API key configured for this runtime.


## Low Confidence HRRSD prediction

In [10]:
!rs-run-assistance-experiment --config configs/evaluation/hrrsd_low_confidence_rescue.yaml

[job] id=20260922T142529Z_fb29355d
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-run-assistance-experiment/20260922T142529Z_fb29355d
Selected cases: 90; already completed: 0; false negatives excluded from per-box VLM evaluation: 3606
[1/90] hrrsd_external_000007_00018_cmp_0001: completed
[2/90] hrrsd_external_000010_00031_cmp_0004: completed
[3/90] hrrsd_external_000025_00161_cmp_0002: completed
[4/90] hrrsd_external_000027_00163_cmp_0003: completed
[5/90] hrrsd_external_000031_00175_cmp_0000: completed
[6/90] hrrsd_external_000038_00215_cmp_0000: completed
[7/90] hrrsd_external_000048_00316_cmp_0001: completed
[8/90] hrrsd_external_000053_00344_cmp_0000: completed
[9/90] hrrsd_external_000054_00347_cmp_0007: completed
[10/90] hrrsd_external_000064_00377_cmp_0001: completed
[11/90] hrrsd_external_000064_00377_cmp_0008: completed
[12/90] hrrsd_external_000064_00377_cmp_0011: completed
[13/90] hrrsd_external_000065_00379_cmp_0005:

In [11]:
!rs-run-assistance-experiment --config configs/evaluation/hrrsd_low_confidence_rescue.yaml

[job] id=20260922T161332Z_bdced557
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-run-assistance-experiment/20260922T161332Z_bdced557
Selected cases: 90; already completed: 90; false negatives excluded from per-box VLM evaluation: 3606
Assistance experiment: /content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/assistance_comparisons/hrrsd_yolov8s1024_low_confidence_rescue_seed42
[job] status=succeeded log=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-run-assistance-experiment/20260922T161332Z_bdced557/run.log


## High Confidence sample

In [12]:
!rs-run-assistance-experiment --config configs/evaluation/hrrsd_vlm_comparison.yaml

[job] id=20260922T161355Z_4c2b36ee
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-run-assistance-experiment/20260922T161355Z_4c2b36ee
Selected cases: 90; already completed: 0; false negatives excluded from per-box VLM evaluation: 3606
[1/90] hrrsd_external_000002_00005_cmp_0003: completed
[2/90] hrrsd_external_000019_00135_cmp_0002: completed
[3/90] hrrsd_external_000044_00260_cmp_0002: completed
[4/90] hrrsd_external_000057_00360_cmp_0000: completed
[5/90] hrrsd_external_000066_00380_cmp_0004: completed
[6/90] hrrsd_external_000073_00395_cmp_0002: completed
[7/90] hrrsd_external_000078_00409_cmp_0001: completed
[8/90] hrrsd_external_000099_00690_cmp_0003: completed
[9/90] hrrsd_external_000148_00843_cmp_0000: completed
[10/90] hrrsd_external_000148_00843_cmp_0001: completed
[11/90] hrrsd_external_000184_00973_cmp_0003: completed
[12/90] hrrsd_external_000184_00973_cmp_0004: completed
[13/90] hrrsd_external_000184_00973_cmp_0005:

In [5]:
import json
from pathlib import Path

experiment_dir = (
    Path("/content/drive/Othercomputers/My laptop/shared_resources")
    / "experiment_outputs"
    / "assistance_comparisons"
    / "hrrsd_yolov8s1024_vlm_high_confidence_seed42"
)

summary = json.loads(
    (experiment_dir / "summary.json").read_text(encoding="utf-8")
)

summary

{'selected_cases': 90,
 'completed_cases': 90,
 'failed_cases': 0,
 'excluded_false_negatives': 3606,
 'status_counts': {'true_positive': 30,
  'false_positive': 30,
  'class_error': 30}}

In [6]:
import pandas as pd

metrics = pd.read_csv(experiment_dir / "metrics.csv")

comparison = metrics.pivot(
    index="metric",
    columns="variant",
    values="value",
)

display(comparison.round(4))

variant,configured_policy,detector_only,detector_retrieval,query_only_vlm,retrieval_grounded_vlm
metric,,,,,
auto_accept_coverage,0.4111,1.0000,0.5222,0.5556,0.5778
auto_accept_precision,0.6216,0.3333,0.6170,0.6000,0.5769
class_correction_accuracy,NaN,NaN,NaN,0.0667,0.1333
decision_accuracy,0.7667,0.3333,0.7889,0.7778,0.7556
false_accept_rate,0.2333,1.0000,0.3000,0.3333,0.3667
hit_at_k,NaN,NaN,0.9833,NaN,NaN
macro_decision_f1,NaN,NaN,NaN,0.3995,0.4047
mean_latency_seconds,NaN,NaN,0.3577,22.6990,31.3798
precision_at_k,NaN,NaN,0.9800,NaN,NaN


In [3]:
!rs-export-predictions --config configs/inference/ultralytics_export.yaml

[job] id=20260806T142521Z_ca65e060
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-export-predictions/20260806T142521Z_ca65e060
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

0: 640x640 (no detections), 0.5ms
1: 640x640 (no detections), 0.5ms
2: 640x640 (no detections), 0.5ms
3: 640x640 1 airplane, 0.5ms
4: 640x640 (no detections), 0.5ms
5: 640x640 (no detections), 0.5ms
6: 640x640 (no detections), 0.5ms
7: 640x640 (no detections), 0.5ms
8: 640x640 (no detections), 0.5ms
9: 640x640 (no detections), 0.5ms
10: 640x640 1 airplane, 0.5ms
11: 640x640 1 baseball_diamond, 0.5ms
12: 640x640 2 airplanes, 0.5ms
13: 640x640 1 basketball_court, 0.5ms
14: 640x640 (no detections), 0.5